## GAN for MNIST

### PART 1: IMPORTS

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

### PART 2: DATASET PREPARATION

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

Purpose: Define how to process images before feeding to network

transforms.Compose([]): Chains multiple transformations together

Step 1: transforms.ToTensor()

Converts PIL Image (0-255) to PyTorch tensor (0-1)

Changes shape from (H, W) to (1, H, W)

Step 2: transforms.Normalize((0.5,), (0.5,))

Formula: output = (input - 0.5) / 0.5

Shifts range from [0, 1] to [-1, 1]

Why? Generator uses Tanh output (range -1 to 1), so inputs should match this range

trainset = datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

In [3]:
trainset = datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

trainloader = DataLoader(
    trainset, 
    batch_size=64, 
    shuffle=True
)



100%|██████████| 9.91M/9.91M [04:16<00:00, 38.6kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 87.2kB/s]
100%|██████████| 1.65M/1.65M [00:12<00:00, 129kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.36MB/s]


### PART 3: GENERATOR ARCHITECTURE

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim=100):               # noise vector is a random vector that serves as input to the generator; The generator learns to transform this noise vector into a realistic image through training, and the noise_dim parameter specifies the dimensionality of this noise vector (i.e., how many random values it contains)
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(True),
            nn.Linear(128, 256),
            nn.ReLU(True),
            nn.Linear(256, 28*28),
            nn.Tanh()
        )
    
    def forward(self, x):
        return self.model(x).view(-1, 1, 28, 28)


What is noise_dim=100?
Think of it like this:

noise_dim = 100 means: 
"Give me 100 random numbers as my inspiration/raw material"
Analogy: The Artist's Sketch Pad
```
Imagine an artist who wants to draw a cat:

Random Scribbles (noise) → Artist's Brain → Finished Drawing
      ↓                         ↓                 ↓
  100 random              Generator           28×28 image
   numbers                (neural net)         of a digit
```     
The artist can't create from NOTHING. They need:
- Some random inspiration
- Some scribbles to start with
- Some "creative chaos" to shape into something meaningful, 
Why 100? Why not 10 or 1000?
```
noise_dim = 10   # Too small: Like giving artist only 10 scribbles
                  # Result: Can only draw 10 variations (boring!)

noise_dim = 100  # Just right: Like giving artist 100 different scribbles
                  # Result: Can draw thousands of unique digits!

noise_dim = 1000 # Too big: Too many scribbles to process
                  # Result: Confuses the artist, harder to learn
```                 
Think of it as "CREATIVE FREEDOM":

Small noise_dim = Less freedom (limited variety)

Large noise_dim = More freedom (more variety, but harder to control)

100 is the "sweet spot" that works well for MNIST

#### Generator Architecture Explained:
```
Layer	Input → Output	Purpose
Linear1	100 → 128	Expand random noise to learnable features
ReLU	128 → 128	Add non-linearity, inplace=True saves memory
Linear2	128 → 256	Further expand, more capacity to learn
ReLU	256 → 256	Non-linearity again
Linear3	256 → 784	Project to image size (28×28 = 784 pixels)
Tanh	784 → 784	Scale outputs to [-1, 1] to match normalized images
```
Why Tanh at the end?

MNIST images are normalized to [-1, 1]

Tanh outputs values in this exact range

Creates better gradients than sigmoid

```
    def forward(self, x):
        return self.model(x).view(-1, 1, 28, 28)
```        
Purpose: Define forward pass

self.model(x): Pass noise through sequential layers

.view(-1, 1, 28, 28): Reshape output to image format

-1: Automatically infer batch size

1: 1 channel (grayscale)

28, 28: Height and width

Shape transformation:
```
Input noise: (batch_size, 100)
After Linear layers: (batch_size, 784)
After view: (batch_size, 1, 28, 28)  ← Ready to display as image!
```

### PART 4: DISCRIMINATOR ARCHITECTURE